# Stage1 分钟现金部署的容量与期限审计

只读封存的 Qlib 分钟标签与逐层点时原账户。检查原 beta 上限下实际可部署权重，并计算事后选择最佳分钟的不可交易上界；零新模型、零候选账户。

In [1]:
import hashlib
import json
import os
from pathlib import Path

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

from src.core.recorder_run import publish_evidence_manifest
from src.core.recorder_sources import load_verified_recorder_objects

root = Path.cwd().resolve()
card = root / "research/stage1_minute_capacity_horizon_audit_20261029"
plan = json.loads((card / "plan.json").read_text())

def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

for key in ("source", "parent"):
    spec = plan[key]
    source_card = root / spec["card"]
    assert sha(source_card / "summary.json") == spec["summary_sha256"]
    assert sha(source_card / "verification_manifest.json") == spec["inventory_sha256"]
assert sha(root / plan["provider_uri"] / "qlib_store_manifest.json") == plan["provider_manifest_sha256"]

qlib.init(provider_uri=str(root / plan["provider_uri"]), region="us", kernels=1)
minute_probe = D.features(["SOXX"], ["$close"], start_time="2022-01-04 09:31:00",
                          end_time="2022-01-04 16:00:00", freq="1min")
assert minute_probe["$close"].notna().sum() >= 300

labels, label_audit = load_verified_recorder_objects(
    root / plan["source"]["card"], expected_recorder_id=plan["source"]["recorder_id"],
    artifacts=["analysis/labels"])
parent, parent_audit = load_verified_recorder_objects(
    root / plan["parent"]["card"], expected_recorder_id=plan["parent"]["recorder_id"],
    artifacts=["analysis/native/dated_f696_0"])
labels = labels["analysis/labels"]
authority = parent["analysis/native/dated_f696_0"]["authority"]
assert len(labels) == plan["expected_minute_slots"]
assert len(authority) == plan["expected_parent_days"]
assert int(labels.funded_change.sum()) == plan["expected_funded_rows"]
assert set(pd.to_datetime(labels.trade_date).unique()) == set(pd.to_datetime(authority.trade_date).unique()) - {
    pd.Timestamp("2022-12-22")
} or set(pd.to_datetime(labels.trade_date).unique()).issubset(set(pd.to_datetime(authority.trade_date).unique()))

funded = labels.loc[labels.funded_change].copy()
funded["trade_date"] = pd.to_datetime(funded.trade_date)
funded = funded.sort_values(["trade_date", "decision_at"], kind="stable")
first = funded.drop_duplicates("trade_date").set_index("trade_date")
groups = funded.groupby("trade_date", sort=True)
perday = pd.DataFrame({
    "first_local_value": first.local_advantage,
    "best_local_value": groups.local_advantage.max(),
    "event_rows": groups.local_advantage.size(),
    "first_cash_weight": first.decision_cash_weight,
    "first_soxl_weight": first.decision_soxl_weight,
    "first_soxx_weight": first.decision_soxx_weight,
    "first_beta_cap": first.current_beta_cap,
    "first_deployment_weight": first.cash_deployment_weight,
})
perday["first_beta_room"] = (perday.first_beta_cap - 3 * perday.first_soxl_weight -
                              perday.first_soxx_weight).clip(lower=0)
perday["first_cash_room"] = (perday.first_cash_weight - 0.15).clip(lower=0)
perday["cap_limited"] = (np.isclose(perday.first_deployment_weight, perday.first_beta_room, atol=1e-8)
    & perday.first_beta_room.lt(perday.first_cash_room - 1e-8)
    & perday.first_beta_room.lt(0.10 - 1e-8))
assert (perday.first_deployment_weight <= perday.first_beta_room + 1e-8).all()
assert (perday.first_deployment_weight <= perday.first_cash_room + 1e-8).all()
assert (perday.first_deployment_weight <= 0.10 + 1e-8).all()
perday["first_or_hold_oracle"] = perday.first_local_value.clip(lower=0)
perday["best_minute_or_hold_oracle"] = perday.best_local_value.clip(lower=0)
perday["year"] = perday.index.year

parent_calendar = pd.to_datetime(authority.trade_date)
annual = []
for year in sorted(parent_calendar.dt.year.unique()):
    g = perday.loc[perday.year.eq(year)]
    annual.append({
        "year": int(year), "parent_days": int(parent_calendar.dt.year.eq(year).sum()),
        "funded_days": int(len(g)), "cap_limited_first_days": int(g.cap_limited.sum()),
        "first_action_under_1pct_days": int(g.first_deployment_weight.lt(0.01).sum()),
        "median_first_cash_weight": float(g.first_cash_weight.median()),
        "median_first_beta_room": float(g.first_beta_room.median()),
        "median_first_deployment_weight": float(g.first_deployment_weight.median()),
        "first_positive_days": int(g.first_local_value.gt(0).sum()),
        "any_positive_minute_days": int(g.best_local_value.gt(0).sum()),
        "first_local_value_sum": float(g.first_local_value.sum()),
        "first_or_hold_oracle_sum": float(g.first_or_hold_oracle.sum()),
        "best_minute_or_hold_oracle_sum": float(g.best_minute_or_hold_oracle.sum()),
    })
annual = pd.DataFrame(annual)
assert annual.funded_days.sum() == len(perday)
assert annual.best_minute_or_hold_oracle_sum.ge(annual.first_or_hold_oracle_sum - 1e-12).all()

summary = {
    "experiment": card.name, "plan_sha256": sha(card / "plan.json"),
    "decision": "read_only_capacity_and_horizon_mismatch_no_promotion",
    "objective_achieved": False, "historical_research_passed": False,
    "parent_days": len(authority), "minute_slots": len(labels), "funded_rows": len(funded),
    "funded_days": len(perday), "annual": annual.to_dict("records"),
    "new_model_fits": 0, "new_native_accounts": 0,
    "provider_uri": str(root / plan["provider_uri"]),
    "source_recorder_ids": {"labels": plan["source"]["recorder_id"], "parent": plan["parent"]["recorder_id"]},
    "source_readback": {"labels": label_audit, "parent": parent_audit},
    "artifact_policy": {"policy": "slim_card_recorder_first", "workflow_config": "workflow.yaml",
                        "recorder_only_artifacts": ["inputs/*", "analysis/*"]},
    "nas_receipt_qualified": False,
}
with R.start(experiment_name=card.name, uri=str(card / "qlib_recorder")):
    rec = R.get_recorder()
    rec.save_objects(artifact_path="inputs", plan=plan)
    rec.save_objects(artifact_path="analysis", perday=perday, annual=annual)
    rec.log_metrics(funded_days=float(len(perday)), cap_limited_2022_days=float(
        annual.loc[annual.year.eq(2022), "cap_limited_first_days"].iloc[0]))
    summary.update(recorder_id=rec.id, experiment_id=str(rec.experiment_id),
        recorder_uri=str(card / "qlib_recorder"), recorder_local_dir=rec.get_local_dir())

publish_evidence_manifest(card, summary)
annual.to_csv(card / "annual_capacity.csv", index=False)
(card / "workflow.yaml").write_text(
    f"experiment: {card.name}\nprovider_uri: {summary['provider_uri']}\n"
    f"recorder_uri: {summary['recorder_uri']}\nartifact_policy:\n"
    "  policy: slim_card_recorder_first\n  workflow_config: workflow.yaml\n"
    "  recorder_only_artifacts:\n  - inputs/*\n  - analysis/*\n")
print(json.dumps({"decision": summary["decision"], "annual": summary["annual"],
                  "recorder_id": summary["recorder_id"]}, ensure_ascii=False, indent=2))


[3816:MainThread](2026-10-29 10:00:07,491) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[3816:MainThread](2026-10-29 10:00:07,492) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[3816:MainThread](2026-10-29 10:00:07,492) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[3816:MainThread](2026-10-29 10:00:08,949) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_minute_capacity_horizon_audit_20261029.


[3816:MainThread](2026-10-29 10:00:08,951) INFO - qlib.workflow - [exp.py:258] - Experiment 622223143721154784 starts running ...


[3816:MainThread](2026-10-29 10:00:09,029) INFO - qlib.workflow - [recorder.py:345] - Recorder ded49b2d8a2647c699999f95df0e5a8c starts running under Experiment 622223143721154784 ...


[3816:MainThread](2026-10-29 10:00:09,714) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{
  "decision": "read_only_capacity_and_horizon_mismatch_no_promotion",
  "annual": [
    {
      "year": 2018,
      "parent_days": 53,
      "funded_days": 45,
      "cap_limited_first_days": 45,
      "first_action_under_1pct_days": 45,
      "median_first_cash_weight": 0.5144773259424588,
      "median_first_beta_room": 0.002271651052417334,
      "median_first_deployment_weight": 0.002271651052417334,
      "first_positive_days": 19,
      "any_positive_minute_days": 36,
      "first_local_value_sum": 1.2695510603123682e-05,
      "first_or_hold_oracle_sum": 0.0008153044579845709,
      "best_minute_or_hold_oracle_sum": 0.009659081075552267
    },
    {
      "year": 2019,
      "parent_days": 252,
      "funded_days": 195,
      "cap_limited_first_days": 192,
      "first_action_under_1pct_days": 164,
      "median_first_cash_weight": 0.21066355860869235,
      "median_first_beta_room": 0.0027696279812417512,
      "median_first_deployment_weight": 0.0027696279812416957,
      "f

# Stage1 分钟现金部署的容量与期限审计

只读封存的逐层点时 f696+b472 账户及 Qlib 一分钟整股含 Tiered 费用标签。1056 个母本交易日、384660 个分钟时点、165695 条实际整股标签逐项与已登记来源一致；本卡零新拟合、零候选账户。逐年完整数字及新 Recorder 身份见 `annual_capacity.csv`、`summary.json`。

| 开发年 | 有整股动作日 | 首次可行动分钟受原 beta 上限约束 | 首次动作权重中位数 | 首次现金权重中位数 | 事后任一分钟为正的日期 |
|---|---:|---:|---:|---:|---:|
| 2021 | 213 | 213 | 0.2231% | 29.87% | 169 |
| 2022 | 180 | 179 | 0.1175% | 70.11% | 127 |

**容量是现有现金直购路线的首要瓶颈。** 固定动作写作“最多部署 10% 现金”，但在 2022 年首次可行动点，179/180 日由原 beta 上限而非现金底线决定动作幅度；169/180 日的动作权重小于 1%。名义“有现金、可买整股”并不意味着可大幅接近 SOXX 暴露。盘中价格变化使部分后续分钟产生更多空间，但原上限仍是该版本必须遵守的约束。这个诊断来自逐笔点时状态，不授权直接放松上限。

**分钟择时有事后空间，却没有可用的信息门。** 2022 年 180 个有整股动作日中，首次合法分钟局部价值为正 83 日；若事后挑选当天任一最佳分钟则有 127 日为正。最佳分钟或 HOLD 的已知结果局部价值和为 +0.020638，首次分钟或 HOLD 为 +0.003425。两者都用了未来标签，是不可交易的乐观上界；不能与已失败模型、原生账户收益或 SOXX 财富直接比较。已封存的两个分钟模型分别在时机和方向信息门失败，不能以这个上界重开或反向调参。

**期限仍未对齐最终目标。** 这些局部标签在下一开盘与母本共同复位，衡量的是一笔短期增量的同状态价值；原 Stage1 与 SOXX 的 2023–2026 缺口包含跨日现金机会成本、SOXL 相对选择和风险状态反馈。2018–2022 开发期按决策 NAV 标准化的局部价值和不可与 2023–2026 持有期对数超额相减。本审计因此确认“分钟现金直购动作幅度被卡住”，但不能证明任何新的风险预算、持有期限或完整账户会胜过 SOXX。

下一条机制须同时检验配置意图与**原风险容量的边际价值**，并把风险约束的好区间保护和坏区间现金拖累放在同一个点时接收任务里。先冻结允许改变的风险预算、整股含费同状态标签、完整开发日历和匹配控制；只有新信息门通过，才运行自身状态 0bp 原生账户。简单扩大 10% 动作、绕过上限、重选分钟窗口或使用事后最佳分钟，都不是合格新机制。

本地来源标签 Recorder 尚无 NAS 工件收据；本卡仅做限定本地哈希读回，新 Recorder 归档状态见 `archive_status.json`。历史分钟首次真实接收未认证。决定 `read_only_capacity_and_horizon_mismatch_no_promotion`，Stage1 SOXX 历史 Goal 未完成。
